## 50_register_models  (NEW workspace) — Model Artifacts Migration step 2 of 3

Video practice (frames 11 & 30): artifacts land in a UC Volume, then each
model is registered into the Unity Catalog Model Registry with an alias.
  volume zip -> unpack -> mlflow.register_model(file:/Volumes/...) -> alias
PREREQ: models_export.zip uploaded into the staging volume (run 45 first).
Widgets: CATALOG/SCHEMA/VOLUME locate the landing zone; MODELS_ZIP names
the uploaded archive. Defaults match 45_setup_new_ws.


In [ ]:

dbutils.widgets.text("CATALOG", "practice_ml", "Landing-zone catalog")
dbutils.widgets.text("SCHEMA", "models", "Landing-zone schema")
dbutils.widgets.text("VOLUME", "staging_artifacts", "Landing-zone volume")
dbutils.widgets.text("MODELS_ZIP", "models_export.zip", "Uploaded zip file name")

CATALOG = dbutils.widgets.get("CATALOG").strip()
SCHEMA = dbutils.widgets.get("SCHEMA").strip()
VOLUME = dbutils.widgets.get("VOLUME").strip()
ZIP_NAME = dbutils.widgets.get("MODELS_ZIP").strip()

VOL_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
ZIP_PATH = f"{VOL_ROOT}/{ZIP_NAME}"
UNPACK_DIR = f"{VOL_ROOT}/models_export"

import os
assert os.path.exists(ZIP_PATH), (
    f"{ZIP_PATH} not found. Upload {ZIP_NAME} into the volume via Catalog Explorer "
    f"(see runbook Phase 2)."
)


In [ ]:

import shutil, pandas as pd

shutil.rmtree(UNPACK_DIR, ignore_errors=True)
shutil.unpack_archive(ZIP_PATH, UNPACK_DIR)

manifest = pd.read_csv(f"{UNPACK_DIR}/model_list.csv")
display(manifest)
assert len(manifest) == 5, f"Expected 5 models in manifest, found {len(manifest)}"


In [ ]:

import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()
results = []

for _, r in manifest.iterrows():
    simple = r["simple_name"]
    src_dir = f"{UNPACK_DIR}/{simple}"
    assert os.path.exists(os.path.join(src_dir, "MLmodel")), f"MLmodel missing in {src_dir}"

    target_name = f"{CATALOG}.{SCHEMA}.{simple}"
    mv = mlflow.register_model(f"file:{src_dir}", target_name)
    client.set_registered_model_alias(target_name, "latest", mv.version)

    results.append({"model": target_name, "version": mv.version, "alias": "latest",
                    "source": f"file:{src_dir}"})
    print(f"REGISTERED  {target_name}  v{mv.version}  alias='latest'")

display(spark.createDataFrame(pd.DataFrame(results)))
print("\nVideo step (frame 30) done: verify in Catalog Explorer ->", f"{CATALOG} -> {SCHEMA}")
print("Then run: 60_verify_models")
